# Loading

In [ ]:
import omicverse as ov
import pandas as pd
import json
import numpy as np
from adjustText import adjust_text
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import anndata as ad
ov.plot_set(font_path='Arial')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42


# Mapping

In [ ]:
sp = sc.read_h5ad("fetal_brain_hongtaoyu/3dCellBin/NB20241129094951824/43_A03590E1G4_WT202403310064.celltype.region.h5ad") 
sp

In [ ]:
sc.pl.spatial(sp,color=['celltype'],spot_size=1)

In [ ]:
adata_GSE76381 = sc.read_h5ad("Process_Data/to_do_list_6/GSE76381.h5ad")
adata_GSE76381

In [ ]:
decov_obj=ov.space.Deconvolution(
    adata_sc=adata_GSE76381,
    adata_sp=sp
)

In [ ]:
decov_obj.preprocess_sc(
    mode='shiftlog|pearson',n_HVGs=10000,target_sum=1e4,
)
decov_obj.preprocess_sp(
    mode='pearsonr',n_svgs=10000,target_sum=1e4,
)

In [ ]:
decov_obj.deconvolution(
    method='Tangram',celltype_key_sc='celltype',
    tangram_kwargs={'mode':'cells','num_epochs':500,'device':'cpu'}
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
sc.pl.spatial(sp,color=['hDA2'],spot_size=1,ax=ax)

# Integration

In [ ]:
adata_GSE76381 = sc.read_h5ad("Process_Data/to_do_list_6/GSE76381.h5ad")
adata_GSE76381 = adata_GSE76381[adata_GSE76381.obs['celltype'].isin(['hDA2','hDA1','hDA0'])]
adata_GSE76381

In [ ]:
adata_GSE76381.obs['batch']= ''

In [ ]:
adata_GSE76381=ov.pp.preprocess(adata_GSE76381,mode='shiftlog|pearson',n_HVGs=120,
                       target_sum=1e4)
adata_GSE76381.raw = adata_GSE76381
adata_GSE76381 = adata_GSE76381[:, adata_GSE76381.var.highly_variable_features]
ov.pp.scale(adata_GSE76381)
ov.pp.pca(adata_GSE76381,layer='scaled',n_pcs=20)

ov.pp.neighbors(adata_GSE76381, n_neighbors=10, n_pcs=20,
               use_rep='scaled|original|X_pca')
ov.pp.umap(adata_GSE76381)
ov.pl.embedding(adata_GSE76381,
                basis='X_umap',
                color=['celltype','ALDH1A1'],
                frameon='small')

In [ ]:
adata_GSE76381 = adata_GSE76381[:, adata_GSE76381.var.highly_variable_features]
adata_GSE76381 = adata_GSE76381.copy()
model=ov.single.batch_correction(adata_GSE76381,batch_key='',
                           methods='scVI',n_layers=2, n_latent=10, gene_likelihood="nb")

ov.pp.neighbors(adata_GSE76381, n_neighbors=15,
               use_rep='X_scVI')
ov.pp.umap(adata_GSE76381)
ov.pl.embedding(adata_GSE76381,
                basis='X_umap',
                color=['celltype','ALDH1A1'],
                frameon='small')